# 1D J1-J2=0.0-J3=0.5: Euclidean GRU

This notebook is part of the work https://arxiv.org/abs/2604.24337. 

In [2]:
import os
import sys
import time
sys.path.append('../utility')
from j1j2j3_hyprnn_train_loop import *

GPU Available:  False


In [3]:
def set_cpu_deterministic(seed=111):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

set_cpu_deterministic(111)

In [6]:
E_exact = -53.99140745384453
units = 80
syssize = 100
nssamples = 80
J1 = 1.0
J2 = 0.0
J3 = 0.5
nsteps = 1201
var_tol = 20.0

# euclGRU

In [4]:
fname=f'1d_J1J2J3_results_N={syssize}/eGRU/J2={J2}_J3={J3}'
eGRU = RNNwavefunction(systemsize=syssize, cell_type='EuclGRU', units=units, seed=111)

for name, param in eGRU.model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()}")
print(f'Total params = {sum(p.numel() for p in eGRU.model.parameters())}')

Layer: rnn.Wz | Size: torch.Size([80, 80])
Layer: rnn.Uz | Size: torch.Size([2, 80])
Layer: rnn.bz | Size: torch.Size([1, 80])
Layer: rnn.Wr | Size: torch.Size([80, 80])
Layer: rnn.Ur | Size: torch.Size([2, 80])
Layer: rnn.br | Size: torch.Size([1, 80])
Layer: rnn.Wh | Size: torch.Size([80, 80])
Layer: rnn.Uh | Size: torch.Size([2, 80])
Layer: rnn.bh | Size: torch.Size([1, 80])
Layer: dense_a.weight | Size: torch.Size([2, 80])
Layer: dense_a.bias | Size: torch.Size([2])
Layer: dense_p.weight | Size: torch.Size([2, 80])
Layer: dense_p.bias | Size: torch.Size([2])
Total params = 20244


In [12]:
t0=time.time()
# since the model converges to very low value of var
var_tol = 1.0
mE, vE = run_J1J2J3(eGRU, nsteps, syssize, var_tol, J1_=J1, J2_=J2, J3_=J3, Marshall_sign=True, 
                   numsamples=nssamples, lr1=5e-3, lr2=1e-3, seed=111, fname=fname)
t1=time.time()
print(f'Time taken = {np.round((t1-t0)/3600,3)} hrs')

step: 0, loss: -1.46826, mean energy: 36.81949-0.04789j, varE: 0.58012, |LR: 5.00e-03
step: 10, loss: 33.96390, mean energy: 3.66711+0.11430j, varE: 38.24837, |LR: 5.00e-03
step: 20, loss: 12.26579, mean energy: -3.53401-0.19200j, varE: 21.63669, |LR: 5.00e-03
step: 30, loss: -17.16968, mean energy: -4.09857+0.05313j, varE: 28.38699, |LR: 5.00e-03
step: 40, loss: 12.98825, mean energy: -7.01986+0.08479j, varE: 29.81080, |LR: 5.00e-03
step: 50, loss: -10.20438, mean energy: -5.35287-0.50671j, varE: 28.75137, |LR: 5.00e-03
step: 60, loss: 10.33917, mean energy: -11.01867-0.30725j, varE: 30.00663, |LR: 5.00e-03
step: 70, loss: -31.18256, mean energy: -19.40700-0.72264j, varE: 34.88633, |LR: 5.00e-03
step: 80, loss: 3.16321, mean energy: -28.15436+0.25175j, varE: 23.41401, |LR: 5.00e-03
step: 90, loss: -9.59473, mean energy: -36.34332+0.18224j, varE: 34.10665, |LR: 5.00e-03
Best model saved at epoch 96 with best E=-37.63997+0.01193j, varE=0.52713
step: 100, loss: -0.49802, mean energy: -37